# Prometheus Star: ARC-AGI-3 Solver (Production Edition)
**Architecture:** Bridge v26 (Null-Move Detection + Scanner Threshold Fix)
**Last updated:** 2026-05-06

This notebook is streamlined for solving the official ARC-AGI-3 benchmark using the latest repository updates.

### v26 Changes (null-move detection + slow-game threshold fix)
- **Null-move game detection** — after 50 move samples, if avg pixels changed
  per move < 1.0 (ACTION1-4 are game no-ops), flag `_null_move_game=True`.
  Nav solver is bypassed entirely; scanner fires from window 1 (no wwr≥4 gate);
  scanner mixes 25% rotate (ACTION5) + 5% undo (ACTION7) + 70% place (ACTION6).
  Targets sb26-type color-sort games where moves are explicitly ignored.
- **Slow-game threshold raised 0.6s → 1.2s** — sk48 (0.62s/step) was being
  incorrectly throttled to 10% scanner after window 47, eliminating all place
  actions for the rest of the game. Only truly slow games like bp35 (~2s/step)
  now trigger the 10% cap.

### v25 Changes (slow-game throttle + inline inspector)
- **Slow-game detection** — on the first scanner window (wwr≥4), measure avg
  step time; if >0.6s flag `_slow_game=True` and cap scanner_prob at 10%.
  Prevents bp35 (~2s/step × 200 steps × 60 windows = 24,000s blowup).
- **Inline inspector** — game-file inspector inlined at the end of the run
  cell so it auto-executes after games complete without needing a manual step.

### v24 Changes (deferred-reward lookback + always-on scanner for click games)
- **30-step lookback** — win capture scans back up to 30 steps from each
  reward to find the triggering place action, fixing lp85 (StepCounter=13
  animation delay between click and reward).
- **`force_scanner`** — once winning place actions are known, scanner bypasses
  the wwr≥4 gate so confirmed click games stay in scan mode every window.

### v23 Changes (win-action replay + systematic param sweep)
- **Win-action replay** — place actions that triggered wins are recorded and
  replayed at the start of every subsequent window, converting one-off lucky
  discoveries into consistent solutions.
- **Deterministic 17-step param cycle** — [None, 0, 1, ..., 15] replaces the
  old 70%/30% random param selection, ensuring full param coverage per cell.

### v22 Changes (nav_dominant detector fixes gravity/physics games)
- **`nav_dominant` signal** — tracks avg pixels changed per move action vs.
  per place action. Physics/gravity games reliably distinguished from click games.
- **Gravity games keep 15% scanner** — `avg_move > 30` and `avg_move > avg_place × 2`
  keeps scanner low so nav dominates for physics games.

In [ ]:
# ── Colab / local setup ──────────────────────
import sys, os

REPO_BRANCH = "claude/arc-agi-3-notebook-XVCzt"   # branch carrying the v18 bridge

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print(f'Cloning Prometheus repository (branch: {REPO_BRANCH})...')
        os.system(f'git clone -b {REPO_BRANCH} https://github.com/pmcray/Prometheus_v0_PoC.git')
    else:
        os.system(f'git -C Prometheus_v0_PoC fetch origin {REPO_BRANCH}')
        os.system(f'git -C Prometheus_v0_PoC checkout {REPO_BRANCH}')
        os.system(f'git -C Prometheus_v0_PoC pull origin {REPO_BRANCH}')

    print('Installing dependencies...')
    os.system('pip install -q arc-agi scikit-learn')
    os.system('pip install -q -e Prometheus_v0_PoC/')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, '..')

print('Prometheus Star v22.0 (nav_dominant Detector)')
print('Last update: 2026-05-04 12:00 UTC')

import json
import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)

# ── WP71 imports ───────────
from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer, ARC3ExplorationPolicy,
    ARC3StrangeLoopAgent, ARC3Benchmark, ARC3BenchmarkReport,
    verify_wp71_exit_criteria, _SyntheticARCGame, _ACTION_TYPES,
)

print('WP71 ARC-AGI-3 module loaded.')
print(f'Canonical action types ({len(_ACTION_TYPES)}): {_ACTION_TYPES}')

### Step 2: API Configuration
To access the full ARC-AGI-3 dataset and benchmark, you need an API key from **[three.arcprize.org](https://three.arcprize.org)**.

1. Log in to [three.arcprize.org](https://three.arcprize.org)
2. Copy your **API Key**
3. In Colab, click the **Secrets** (key icon) on the left sidebar
4. Add a new secret with name `ARC_API_KEY` and paste your key as the value
5. Enable the **Notebook access** toggle for this secret

In [ ]:
# Cell 2: API Configuration & Bridge Loading
import os
try:
    from google.colab import userdata
    ARC_API_KEY = userdata.get("ARC_API_KEY")
    os.environ["ARC_API_KEY"] = ARC_API_KEY
    if ARC_API_KEY:
        print(f"API Key active ({ARC_API_KEY[:6]}...)")
    else:
        print("No API Key found - anonymous access enabled.")
except:
    ARC_API_KEY = None
    print("No API Key found - anonymous access enabled.")

from prometheus.arc3_bridge import *
# Weight auto-load is now handled inside run_live_game() — this call is
# idempotent and only populates the in-memory transformer if a checkpoint exists.
load_vision_weights()
print(f"Bridge v26 loaded (Toolkit Available: {TOOLKIT_AVAILABLE}).")
print("Object-centric modules:",
      "ObjectExtractor" in dir(), "GameTypeClassifier" in dir())

def visualize_arc3_episode(episode, max_steps=10):
    """Visualise the first N steps of an ARC-AGI-3 episode."""
    steps = min(len(episode.history), max_steps)
    if steps == 0: return

    fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2))
    if steps == 1: axes = [axes]

    for i in range(steps):
        obs, action, reward = episode.history[i]
        grid = np.array(obs.grid)
        axes[i].imshow(grid, cmap='tab20', vmin=0, vmax=15)
        axes[i].set_title(f"S{i}: {action.action_type}\nR={reward:.1f}", fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Run Prometheus on ARC-AGI-3 ──────
#
# Bridge v26: null-move game detection (sb26-type games where ACTION1-4 are
# no-ops) fires scanner+rotate+undo from window 1, bypassing the nav solver.
# Slow-game threshold raised 0.6s → 1.2s to stop sk48 from being incorrectly
# throttled (was suppressing all place actions after window 47).

SELECTED_GAMES = ["ls20", "ft09", "vc33"]
if ARC_API_KEY and TOOLKIT_AVAILABLE:
    try:
        arc = arc_agi.Arcade()
        all_envs = arc.get_environments()
        SELECTED_GAMES = [e.game_id for e in all_envs]
        print(f"API Key detected: Loading all {len(SELECTED_GAMES)} games.")
    except:
        print("API Key failed: Falling back to public games.")

N_WINDOWS     = 60
WINDOW_STEPS  = 200

live_results = {}
all_episodes = []
t_start = time.time()

# Run on first 5 games to keep demo snappy if all loaded
GAMES_TO_RUN = SELECTED_GAMES[:5] if len(SELECTED_GAMES) > 3 else SELECTED_GAMES

for game_id in GAMES_TO_RUN:
    print()
    print('=' * 55)
    print(f"  Game: {game_id}")
    print('=' * 55)
    result = run_live_game(game_id=game_id, n_windows=N_WINDOWS, window_steps=WINDOW_STEPS, mutation_rate=0.10, fitness_threshold=0.5, verbose=True)
    if result:
        live_results[game_id] = result
        if result.get('last_episode'):
            all_episodes.append(result['last_episode'])

        sr_val = result.get('solve_rate', 0)
        ms_val = result.get('mean_score', 0)
        fam   = result.get('game_family', 'unknown')
        print(f"  --> solve rate: {sr_val:.0%}  mean score: {ms_val:.3f}  family: {fam}")

        # Visualize the last episode
        if result.get('last_episode'):
            print(f"  Visualising last window of {game_id}...")
            visualize_arc3_episode(result['last_episode'], max_steps=8)

        # Step 5: Persistence (run_live_game already saves, this is belt-and-braces)
        save_vision_weights()

print()
print(f"Total time: {time.time()-t_start:.1f}s")

# Step 3: Latent Space Visualization
if all_episodes:
    print("Plotting Latent Space (PCA projection of Transformer embeddings)...")
    visualize_latent_space(all_episodes)

# ── Inline Game Inspector (auto-runs after games complete) ────────────────
import glob, re as _re
_env_base = 'environment_files'
if not os.path.exists(_env_base):
    _env_base = '/content/Prometheus_v0_PoC/environment_files'
_game_files = sorted(glob.glob(f'{_env_base}/**/*.py', recursive=True))
print(f'\nGame files on disk: {len(_game_files)}\n')
for _gf in _game_files:
    print('=' * 70)
    print(f'FILE: {_gf}')
    print('=' * 70)
    with open(_gf) as _f:
        _cnt = _f.read()
    _sm = _re.search(r'sprites\s*=\s*\{', _cnt)
    if _sm:
        _d, _i = 0, _sm.end() - 1
        while _i < len(_cnt):
            if _cnt[_i] == '{': _d += 1
            elif _cnt[_i] == '}':
                _d -= 1
                if _d == 0: _i += 1; break
            _i += 1
        _ns = _cnt[_sm.start():_i].count('Sprite(')
        _cl = _cnt[:_sm.start()] + f'# [SPRITES: {_ns} omitted]\n' + _cnt[_i:]
    else:
        _cl = _cnt
    print(_cl[:8000])
    if len(_cl) > 8000:
        print(f'... [{len(_cl)-8000} chars truncated]')
    for _kw in ['def step(', 'def handle_action(', 'def complete(', 'def _win', 'levels_completed']:
        _ix = _cl.find(_kw)
        if _ix == -1: continue
        print(f'\n--- {_kw} ---')
        print(_cl[_ix:_ix+800])
    print()

In [ ]:
# ── Game File Inspector v2 ──────────────────────────────────────────────────
# Extracts the game-class logic (skipping the bulky `sprites = {...}` block).
# Run AFTER Cell 3.
import os, glob, re

env_base = 'environment_files'
if not os.path.exists(env_base):
    env_base = '/content/Prometheus_v0_PoC/environment_files'

game_files = sorted(glob.glob(f'{env_base}/**/*.py', recursive=True))
print(f'Found {len(game_files)} game file(s)\n')

for gf in game_files:
    print('=' * 70)
    print(f'FILE: {gf}')
    print('=' * 70)
    with open(gf) as f:
        content = f.read()

    # 1) Strip the giant sprites = { ... } literal so we can see the logic.
    sprites_match = re.search(r'sprites\s*=\s*\{', content)
    if sprites_match:
        depth, i = 0, sprites_match.end() - 1
        while i < len(content):
            if content[i] == '{': depth += 1
            elif content[i] == '}':
                depth -= 1
                if depth == 0:
                    i += 1; break
            i += 1
        n_sprites = content[sprites_match.start():i].count('Sprite(')
        content_logic = content[:sprites_match.start()] + f'# [SPRITES BLOCK: {n_sprites} sprites omitted]\n' + content[i:]
    else:
        content_logic = content

    # 2) Print the surviving logic (cap at 10000 chars per file).
    if len(content_logic) > 10000:
        print(content_logic[:10000])
        print(f'... [{len(content_logic)-10000} chars truncated]')
    else:
        print(content_logic)

    # 3) Extract win-condition method bodies (step, handle_action, complete, solved).
    win_kws = ['def step(', 'def handle_action(', 'def on_action(', 'def is_solved',
               'def is_complete', 'levels_completed', 'def _check', 'def check_win',
               'def complete(', 'def _win', 'def solve']
    found_any = False
    for kw in win_kws:
        idx = content_logic.find(kw)
        if idx == -1:
            continue
        if not found_any:
            print('\n' + '-'*40 + '  WIN-CONDITION METHODS  ' + '-'*40)
            found_any = True
        # Print up to 1200 chars from the method start
        snippet = content_logic[idx:idx+1200]
        print(f'\n--- {kw} (char {idx}) ---')
        print(snippet)
    print()
